In [ ]:
pip install ultralytics

In [ ]:
import wandb
from google.colab import userdata
import os

# os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API")
# wandb.login()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix
)
import random
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from PIL import Image
import wandb
import copy
import timm
from ultralytics import YOLO
import time

In [ ]:
CONFIG = {
    # Paths
    'DATASET_PATH': '/content/drive/MyDrive/Dataset/castor_v2_224x224',
    'YOLO_WEIGHTS': '/content/drive/MyDrive/Models_Trained/YOLOv8_YOLOV8_Original_Kaggle_fold1.pth',  # Your trained YOLOv8 classifier weights
    'VIT_WEIGHTS': '/content/drive/MyDrive/Models_Trained/ViTPretrainedReducedFreeze_Improved_NoLabelSmoothing_fold5.pth',

    # Training
    'BATCH_SIZE': 32,
    'NUM_EPOCHS': 100,
    'LEARNING_RATE': 1e-3,
    'WEIGHT_DECAY': 0.01,
    'DROPOUT': 0.3,

    # K-Fold
    'K_FOLDS': 5,
    'RANDOM_STATE': 42,

    # Model
    'NUM_CLASSES': 3,
    'IMG_SIZE': 224,

    # W&B
    'PROJECT_NAME': 'YOLOViT_Fusion',
    "use_wandb": False
}

RANDOM_STATE = CONFIG['RANDOM_STATE']

In [ ]:
RANDOM_STATE = 42

# Ensure deterministic behavior
torch.backends.cudnn.deterministic = True
random.seed(hash("setting random seeds") % 2**32 - 1)
np.random.seed(hash("improves reproducibility") % 2**32 - 1)
torch.manual_seed(hash("by removing stochasticity") % 2**32 - 1)
torch.cuda.manual_seed_all(hash("so runs are repeatable") % 2**32 - 1)

# Device configuration

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
# ============================================================================
# DATASET
# ============================================================================
class ImageDataset(Dataset):
    """Dataset for loading images from folder structure"""
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label


def get_dataset():
    """Load dataset from folder structure"""
    dataset_path = CONFIG['DATASET_PATH']

    image_paths = []
    labels = []
    class_names = sorted(os.listdir(dataset_path))
    class_to_idx = {name: idx for idx, name in enumerate(class_names)}

    print(f"Found classes: {class_names}")

    for class_name in class_names:
        class_dir = os.path.join(dataset_path, class_name)
        if not os.path.isdir(class_dir):
            continue

        for img_name in os.listdir(class_dir):
            img_path = os.path.join(class_dir, img_name)
            if img_path.lower().endswith(('.png', '.jpg', '.jpeg')):
                image_paths.append(img_path)
                labels.append(class_to_idx[class_name])

    print(f"Total images: {len(image_paths)}")
    print(f"Class distribution: {np.bincount(labels)}")

    # Create transform
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])

    # Create dataset
    dataset = ImageDataset(np.array(image_paths), np.array(labels), transform)
    return dataset, class_names

In [ ]:
# ============================================================================
# YOLO MODEL COMPONENTS
# ============================================================================

class YOLOv8Classifier(nn.Module):
    def __init__(self, num_classes=3, pretrained=True):
        super(YOLOv8Classifier, self).__init__()

        if pretrained:
            yolo_model = YOLO('yolov8n.pt')
        else:
            yolo_model = YOLO('yolov8n.yaml')

        self.model = yolo_model.model.model
        self.backbone_neck_layers = nn.ModuleList(list(self.model.children())[:-1])

        # Identify skip-connection layers
        self.save_indices = []
        for i, layer in enumerate(self.model.children()):
            if hasattr(layer, 'f') and layer.f != -1:
                if isinstance(layer.f, int):
                    self.save_indices.append(layer.f)
                elif isinstance(layer.f, (list, tuple)):
                    self.save_indices.extend(layer.f)

        # Infer output channels
        with torch.no_grad():
            dummy_input = torch.randn(1, 3, 640, 640)
            features = self._extract_features(dummy_input)
            if isinstance(features, (list, tuple)):
                features = features[-1]
            self.feature_channels = features.shape[1]

        # Classification head
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(self.feature_channels, num_classes)
        )

        # Freeze backbone initially
        for param in self.backbone_neck_layers.parameters():
            param.requires_grad = False

    def _extract_features(self, x):
        outputs = []
        for i, layer in enumerate(self.backbone_neck_layers):
            if hasattr(layer, 'f'):
                if layer.f != -1:
                    if isinstance(layer.f, int):
                        x = layer(outputs[layer.f]) if layer.f != -1 else layer(x)
                    else:
                        feature_list = [outputs[j] if j >= 0 else x for j in layer.f]
                        x = layer(feature_list)
                        if isinstance(x, (list, tuple)):
                            x = x[-1]
                else:
                    x = layer(x)
            else:
                x = layer(x)
            outputs.append(x)
        return x

    def forward(self, x):
        features = self._extract_features(x)
        if isinstance(features, (list, tuple)):
            features = features[-1]
        output = self.classifier(features)
        return output

    def unfreeze_backbone(self):
        for param in self.backbone_neck_layers.parameters():
            param.requires_grad = True
class YOLOv8Backbone(nn.Module):
    """Extract features from YOLOv8 Classifier"""
    def __init__(self, model):
        super().__init__()
        self.detector_layers = model.backbone_neck_layers
        self.classifier_pool = model.classifier  # Use classifier's pooling

        # Determine output dimension dynamically by running a dummy forward pass
        try:
            # Build a dummy input on the same device as the model
            device_ = next(model.parameters()).device
            img_size = CONFIG.get('IMG_SIZE', 224)
            dummy = torch.zeros(1, 3, img_size, img_size, device=device_)

            self.eval()
            with torch.no_grad():
                feats = self._extract_features(dummy)
                pooled = self.classifier_pool[0](feats)  # AdaptiveAvgPool2d
                pooled = self.classifier_pool[1](pooled)  # Flatten

                self.out_dim = pooled.shape[1]
        except Exception as e:
            # Fallback to configured linear in_features if available
            if hasattr(model.classifier[2], 'in_features'):
                self.out_dim = model.classifier[2].in_features
            else:
                raise RuntimeError(f"Unable to infer YOLO backbone output dimension: {e}")

    def _extract_features(self, x):
        """Extract features from YOLO backbone"""
        outputs = []
        for i, layer in enumerate(self.detector_layers):
            if hasattr(layer, 'f'):
                if layer.f != -1:
                    if isinstance(layer.f, int):
                        x = layer(outputs[layer.f])
                    else:
                        feature_list = [outputs[j] if j >= 0 else x for j in layer.f]
                        x = layer(feature_list)
                        if isinstance(x, (list, tuple)):
                            x = x[-1]
                else:
                    x = layer(x)
            else:
                x = layer(x)
            outputs.append(x)
        return x

    def forward(self, x):
        # Extract backbone features
        features = self._extract_features(x)

        # Apply pooling from classifier (without final linear layer)
        # Get features before the final linear layer
        pooled = self.classifier_pool[0](features)  # AdaptiveAvgPool2d
        pooled = self.classifier_pool[1](pooled)  # Flatten

        return pooled


In [ ]:
# ============================================================================
# VIT MODEL COMPONENTS
# ============================================================================

def build_vit_tiny(num_classes, num_blocks_to_keep=8):
    """Build ViT-Tiny model"""
    vit = timm.create_model('vit_tiny_patch16_224', pretrained=False, num_classes=0)
    embed_dim = 192

    # Reduce blocks
    if num_blocks_to_keep < len(vit.blocks):
        vit.blocks = vit.blocks[:num_blocks_to_keep]

    class ViTWithHead(nn.Module):
        def __init__(self, vit_backbone, embed_dim, num_classes, dropout=0.1):
            super().__init__()
            self.vit = vit_backbone
            self.head = nn.Sequential(
                nn.LayerNorm(embed_dim),
                nn.Dropout(dropout),
                nn.Linear(embed_dim, num_classes)
            )

        def forward(self, x):
            x = self.vit.forward_features(x)
            cls_token = x[:, 0]
            logits = self.head(cls_token)
            return logits

    return ViTWithHead(vit, embed_dim, num_classes)


# ============================================================================
# BACKBONE EXTRACTORS
# ============================================================================

class ViTTinyBackbone(nn.Module):
    """Extract CLS token from ViT-Tiny"""
    def __init__(self, model):
        super().__init__()
        self.vit = model.vit
        self.out_dim = 192

    def forward(self, x):
        features = self.vit.forward_features(x)
        cls_token = features[:, 0]
        return cls_token

In [ ]:
# ============================================================================
# FUSION MODEL
# ============================================================================

class YOLOViTFusion(nn.Module):
    """Fusion model combining YOLOv8 and ViT-Tiny"""
    def __init__(self, yolo_backbone, vit_backbone, num_classes, dropout=0.3):
        super().__init__()
        self.yolo = yolo_backbone
        self.vit = vit_backbone

        fused_dim = self.yolo.out_dim + self.vit.out_dim

        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),

            nn.Linear(256, num_classes)
        )

        self._init_classifier()

    def _init_classifier(self):
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        with torch.no_grad():
            f_yolo = self.yolo(x)
            f_vit = self.vit(x)

        fused = torch.cat([f_yolo, f_vit], dim=1)
        out = self.classifier(fused)
        return out

    def get_num_params(self, trainable_only=False):
        if trainable_only:
            return sum(p.numel() for p in self.parameters() if p.requires_grad)
        return sum(p.numel() for p in self.parameters())

In [ ]:
# ============================================================================
# LOAD PRETRAINED BACKBONES
# ============================================================================

def load_pretrained_backbones():
    """Load pretrained models and create frozen backbones"""
    num_classes = CONFIG['NUM_CLASSES']

    # Build YOLOv8 model
    print("Building YOLOv8 Classifier...")
    yolo_model = YOLOv8Classifier(num_classes=num_classes, pretrained=True)

    # Load trained weights
    print("Loading YOLOv8 weights...")
    yolo_checkpoint = torch.load(CONFIG['YOLO_WEIGHTS'], map_location=device)

    # Extract state dict
    if isinstance(yolo_checkpoint, dict):
        if 'model_state_dict' in yolo_checkpoint:
            state_dict = yolo_checkpoint['model_state_dict']
        elif 'state_dict' in yolo_checkpoint:
            state_dict = yolo_checkpoint['state_dict']
        else:
            state_dict = yolo_checkpoint
    else:
        state_dict = yolo_checkpoint

    # Remap keys: "model.X" -> "backbone_neck_layers.X"
    # The saved weights have "model." prefix, but YOLOv8Classifier expects "backbone_neck_layers."
    new_state_dict = {}
    for key, value in state_dict.items():
        if key.startswith('model.'):
            # Extract the layer number and rest of the key
            # e.g., "model.0.conv.weight" -> "backbone_neck_layers.0.conv.weight"
            new_key = key.replace('model.', 'backbone_neck_layers.', 1)
            new_state_dict[new_key] = value
        else:
            # Keep other keys as is (like classifier weights)
            new_state_dict[key] = value

    # Load the remapped state dict
    yolo_model.load_state_dict(new_state_dict, strict=False)
    print(f"✓ Loaded YOLO weights (remapped {len(new_state_dict)} keys)")

    # Build ViT model
    print("Building ViT-Tiny...")
    vit_model = build_vit_tiny(num_classes, num_blocks_to_keep=8)

    # Load trained weights
    print("Loading ViT-Tiny weights...")
    vit_checkpoint = torch.load(CONFIG['VIT_WEIGHTS'], map_location=device)

    if isinstance(vit_checkpoint, dict):
        if 'model_state_dict' in vit_checkpoint:
            vit_model.load_state_dict(vit_checkpoint['model_state_dict'])
        elif 'state_dict' in vit_checkpoint:
            vit_model.load_state_dict(vit_checkpoint['state_dict'])
        else:
            vit_model.load_state_dict(vit_checkpoint)
    else:
        vit_model.load_state_dict(vit_checkpoint)

    # Move to device and set to eval
    yolo_model.to(device).eval()
    vit_model.to(device).eval()

    # Create backbones
    yolo_backbone = YOLOv8Backbone(yolo_model)
    vit_backbone = ViTTinyBackbone(vit_model)

    # Freeze parameters
    for p in yolo_backbone.parameters():
        p.requires_grad = False
    for p in vit_backbone.parameters():
        p.requires_grad = False

    print("✓ Backbones loaded and frozen!")
    print(f"YOLO feature dim: {yolo_backbone.out_dim}")
    print(f"ViT feature dim: {vit_backbone.out_dim}")

    return yolo_backbone, vit_backbone


# ============================================================================
# TRAINING UTILITIES
# ============================================================================

def plot_confusion_matrix(cm, class_names, title):
    """Plot confusion matrix and save to file"""
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'label': 'Count'})

    ax.set_ylabel("Actual")
    ax.set_xlabel("Predicted")
    ax.set_title(title)

    img_path = f"{title.replace(' ', '_')}.png"
    plt.savefig(img_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return img_path


def save_model_locally(model_state_dict, fold_idx, group_name):
    """
    Save model checkpoint locally only (no W&B artifact upload)
    """
    save_path = f"YOLOViT_{group_name}_fold{fold_idx}.pt"
    try:
        torch.save(model_state_dict, save_path)
        print(f"✓ Saved checkpoint to {save_path}")
    except Exception as e:
        print(f"⚠️ Error saving model locally: {e}")
    return save_path


# ============================================================================
# TRAIN FOLD
# ============================================================================

def train_fold(fold_idx, train_loader, val_loader, group_name, class_names):
    """Trains a single fold"""

    # CRITICAL: Ensure clean W&B state
    if CONFIG.get("use_wandb"):
        try:
            wandb.finish()  # Close any lingering runs
        except:
            pass

    # Small delay to ensure clean state
    time.sleep(2)

    # Now initialize run
    if CONFIG["use_wandb"]:
      run = wandb.init(
          project=CONFIG["PROJECT_NAME"],
          group=group_name,
          name=f"fold-{fold_idx}",
          job_type="train",
          config=CONFIG,
          reinit=True,
          settings=wandb.Settings(
              start_method="thread",  # Better for notebooks
              _disable_stats=False,
              _disable_meta=False
          )
      )

      print(f"✓ W&B run initialized: {run.name} ({run.id})")

    # Load backbones and create model
    yolo_backbone, vit_backbone = load_pretrained_backbones()
    model = YOLOViTFusion(
        yolo_backbone,
        vit_backbone,
        CONFIG['NUM_CLASSES'],
        dropout=CONFIG['DROPOUT']
    ).to(device)

    # Print detailed model info
    print(f"\n{'='*60}")
    print("MODEL ARCHITECTURE SUMMARY")
    print(f"{'='*60}")

    # Count parameters for each component
    yolo_params = sum(p.numel() for p in model.yolo.parameters())
    yolo_trainable = sum(p.numel() for p in model.yolo.parameters() if p.requires_grad)

    vit_params = sum(p.numel() for p in model.vit.parameters())
    vit_trainable = sum(p.numel() for p in model.vit.parameters() if p.requires_grad)

    classifier_params = sum(p.numel() for p in model.classifier.parameters())
    classifier_trainable = sum(p.numel() for p in model.classifier.parameters() if p.requires_grad)

    total_params = model.get_num_params()
    total_trainable = model.get_num_params(trainable_only=True)

    print(f"\nYOLO Backbone:")
    print(f"  Total parameters:     {yolo_params:>12,}")
    print(f"  Trainable parameters: {yolo_trainable:>12,}")
    print(f"  Status: {'FROZEN ❄️' if yolo_trainable == 0 else 'TRAINABLE '}")

    print(f"\nViT Backbone:")
    print(f"  Total parameters:     {vit_params:>12,}")
    print(f"  Trainable parameters: {vit_trainable:>12,}")
    print(f"  Status: {'FROZEN ❄️' if vit_trainable == 0 else 'TRAINABLE '}")

    print(f"\nFusion Classifier:")
    print(f"  Total parameters:     {classifier_params:>12,}")
    print(f"  Trainable parameters: {classifier_trainable:>12,}")

    print(f"\nOVERALL MODEL:")
    print(f"  Total parameters:     {total_params:>12,}")
    print(f"  Trainable parameters: {total_trainable:>12,}")
    print(f"  Frozen parameters:    {total_params - total_trainable:>12,}")
    print(f"  Trainable ratio:      {100 * total_trainable / total_params:>11.2f}%")
    print(f"{'='*60}\n")

    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        model.parameters(),
        lr=CONFIG["LEARNING_RATE"],
        weight_decay=CONFIG["WEIGHT_DECAY"]
    )

    # Scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        patience=3,
        factor=0.3,
    )

    # Training tracking
    best_val_loss = np.inf
    best_weights = None
    patience = 8
    patience_counter = 0
    best_epoch = -1

    # Training loop
    all_train_losses = []
    all_val_losses = []

    for epoch in range(CONFIG["NUM_EPOCHS"]):
        # ------------------- TRAIN -------------------
        model.train()
        # Keep backbones in eval mode
        model.yolo.eval()
        model.vit.eval()

        train_loss = 0.0
        correct_train = 0
        total_train = 0
        all_train_labels = []
        all_train_preds = []
        all_train_probs = []

        progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['NUM_EPOCHS']}")

        for batch_X, batch_y in progress:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()

            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item()
            probs_train = torch.softmax(outputs, dim=1)
            _, predicted_train = torch.max(outputs.data, 1)
            total_train += batch_y.size(0)
            correct_train += (predicted_train == batch_y).sum().item()

            all_train_labels.append(batch_y.cpu().numpy())
            all_train_preds.append(predicted_train.cpu().numpy())
            all_train_probs.append(probs_train.detach().cpu().numpy())

        avg_train_loss = train_loss / len(train_loader)
        train_accuracy = correct_train / total_train
        all_train_losses.append(avg_train_loss)

        # Calculate train metrics
        all_train_labels = np.concatenate(all_train_labels)
        all_train_preds = np.concatenate(all_train_preds)
        all_train_probs = np.concatenate(all_train_probs)

        train_precision = precision_score(all_train_labels, all_train_preds, average='macro', zero_division=0)
        train_recall = recall_score(all_train_labels, all_train_preds, average='macro', zero_division=0)

        try:
            train_roc_auc = roc_auc_score(all_train_labels, all_train_probs, multi_class="ovr", average="macro")
        except:
            train_roc_auc = float("nan")

        # ------------------- VALIDATE -------------------
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        all_labels = []
        all_preds = []
        all_probs = []

        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X)

                loss = criterion(outputs, batch_y)
                val_loss += loss.item()

                probs = torch.softmax(outputs, dim=1)
                _, predicted = torch.max(outputs.data, 1)

                total += batch_y.size(0)
                correct += (predicted == batch_y).sum().item()

                all_labels.append(batch_y.cpu().numpy())
                all_preds.append(predicted.cpu().numpy())
                all_probs.append(probs.cpu().numpy())

        avg_val_loss = val_loss / len(val_loader)
        val_accuracy = correct / total
        all_val_losses.append(avg_val_loss)

        all_labels = np.concatenate(all_labels)
        all_preds = np.concatenate(all_preds)
        all_probs = np.concatenate(all_probs)

        precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
        recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)

        try:
            roc_auc = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")
        except:
            roc_auc = float("nan")

        # ------------------- CALLBACKS -------------------
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            best_epoch = epoch + 1
            best_weights = copy.deepcopy(model.state_dict())
            print(f"   ✔ New best model found at epoch {best_epoch} (val_loss={best_val_loss:.4f})")
        else:
            patience_counter += 1
            print(f"   patience: {patience_counter}/{patience}")

        if patience_counter >= patience:
            print("⛔ Early stopping triggered — patience exceeded")
            break

        scheduler.step(avg_val_loss)

        # ------------------- W&B LOG -------------------
        if CONFIG.get("use_wandb"):
            wandb.log({
                "epoch": epoch + 1,
                "train_loss": avg_train_loss,
                "train_accuracy": train_accuracy,
                "train_precision": train_precision,
                "train_recall": train_recall,
                "train_roc_auc": train_roc_auc,
                "val_loss": avg_val_loss,
                "val_accuracy": val_accuracy,
                "val_precision": precision,
                "val_recall": recall,
                "val_roc_auc": roc_auc,
                "current_lr": optimizer.param_groups[0]["lr"],
                "fold": fold_idx
            })

        print(f"Fold {fold_idx} | Epoch {epoch+1} | "
              f"Train Acc: {train_accuracy:.4f} | Val Acc: {val_accuracy:.4f} | "
              f"Prec: {precision:.4f} | Rec: {recall:.4f} | ROC-AUC: {roc_auc:.4f} | "
              f"LR: {optimizer.param_groups[0]['lr']:.6f}")

    # ------------------- FINAL EVAL WITH BEST MODEL -------------------
    if best_weights is None:
        raise RuntimeError("No best_weights found")

    model.load_state_dict(best_weights)
    print(f"✔ Loaded best model weights for fold {fold_idx} (best epoch = {best_epoch})")

    # Final evaluation
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)

            loss = criterion(outputs, batch_y)
            val_loss += loss.item()

            probs = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs.data, 1)

            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()

            all_labels.append(batch_y.cpu().numpy())
            all_preds.append(predicted.cpu().numpy())
            all_probs.append(probs.cpu().numpy())

    final_avg_val_loss = val_loss / len(val_loader)
    final_val_accuracy = correct / total

    all_labels = np.concatenate(all_labels)
    all_preds = np.concatenate(all_preds)
    all_probs = np.concatenate(all_probs)

    final_precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    final_recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)

    try:
        final_roc_auc = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")
    except:
        final_roc_auc = float("nan")

    final_cm = confusion_matrix(all_labels, all_preds)

    # ------------------- SAVE MODEL -------------------
    print("\n📦 Saving model...")

    save_path = save_model_locally(
        best_weights,
        fold_idx,
        group_name
    )

    # ------------------- LOG CONFUSION MATRIX -------------------
    img_path = plot_confusion_matrix(
        final_cm,
        class_names,
        f"Fold{fold_idx}_Best_Validation_CM"
    )
    if CONFIG.get("use_wandb"):
        wandb.log({
            "best_model_confusion_matrix": wandb.Image(img_path),
            "final_metrics": {
                "accuracy": final_val_accuracy,
                "precision": final_precision,
                "recall": final_recall,
                "roc_auc": final_roc_auc
            }
        })

        # ------------------- SET SUMMARY -------------------
        wandb.summary["fold"] = fold_idx
        wandb.summary["best_epoch"] = best_epoch
        wandb.summary["best_val_loss"] = best_val_loss
        wandb.summary["final_val_loss"] = final_avg_val_loss
        wandb.summary["final_val_accuracy"] = final_val_accuracy
        wandb.summary["final_precision"] = final_precision
        wandb.summary["final_recall"] = final_recall
        wandb.summary["final_roc_auc"] = final_roc_auc

        # CRITICAL: Ensure everything is uploaded before finishing
        print("⏳ Finalizing W&B run...")
        time.sleep(3)  # Give W&B time to process all logs
        wandb.finish()

    # Clean up local checkpoint (optional)
    # os.remove(save_path)

    return {
        'fold': fold_idx,
        'best_epoch': best_epoch,
        'val_accuracy': final_val_accuracy,
        'precision': final_precision,
        'recall': final_recall,
        'roc_auc': final_roc_auc,
        'train_losses': all_train_losses,
        'val_losses': all_val_losses,
        'model_path': save_path
    }


# ============================================================================
# MAIN K-FOLD LOOP
# ============================================================================

def run_k_fold():
    """Run K-Fold cross-validation"""
    suffix = "Original"
    experiment_group_name = f"{suffix}"
    print(f"Starting K-Fold Experiment: {experiment_group_name}")

    # Load dataset
    dataset, class_names = get_dataset()
    print(f"Class names: {class_names}")

    # K-Fold splitter
    kf = StratifiedKFold(n_splits=CONFIG["K_FOLDS"], shuffle=True, random_state=RANDOM_STATE)

    # Get labels for stratification
    labels = []
    for _, label in dataset:
        labels.append(label)
    labels = np.array(labels)

    # Store results
    all_results = []

    # K-Fold loop
    for fold_idx, (train_indices, val_indices) in enumerate(kf.split(np.arange(len(dataset)), labels)):
        print(f"\n{'='*60}")
        print(f"Starting Fold {fold_idx + 1}/{CONFIG['K_FOLDS']}")
        print(f"{'='*60}")

        # Create subsets
        train_subset = Subset(dataset, train_indices)
        val_subset = Subset(dataset, val_indices)

        # Create dataloaders
        train_loader = DataLoader(
            train_subset,
            batch_size=CONFIG["BATCH_SIZE"],
            shuffle=True,
            num_workers=2,
            pin_memory=True
        )
        val_loader = DataLoader(
            val_subset,
            batch_size=CONFIG["BATCH_SIZE"],
            shuffle=False,
            num_workers=2,
            pin_memory=True
        )

        # Train fold
        fold_result = train_fold(
            fold_idx + 1,
            train_loader,
            val_loader,
            experiment_group_name,
            class_names
        )
        all_results.append(fold_result)

    # ------------------- SUMMARY -------------------
    print(f"\n{'='*60}")
    print("K-FOLD CROSS VALIDATION SUMMARY")
    print(f"{'='*60}")

    accuracies = [r['val_accuracy'] for r in all_results]
    precisions = [r['precision'] for r in all_results]
    recalls = [r['recall'] for r in all_results]
    roc_aucs = [r['roc_auc'] for r in all_results]

    print(f"\n AGGREGATED METRICS:")
    print(f"Accuracy:  {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
    print(f"Precision: {np.mean(precisions):.4f} ± {np.std(precisions):.4f}")
    print(f"Recall:    {np.mean(recalls):.4f} ± {np.std(recalls):.4f}")
    print(f"ROC-AUC:   {np.mean(roc_aucs):.4f} ± {np.std(roc_aucs):.4f}")

    print(f"\n PER-FOLD RESULTS:")
    for i, result in enumerate(all_results):
        f1 = (result['precision'] + result['recall']) / 2
        print(f"Fold {result['fold']}: Acc={result['val_accuracy']:.4f}, "
              f"Prec={result['precision']:.4f}, Rec={result['recall']:.4f}, "
              f"F1={f1:.4f}, ROC-AUC={result['roc_auc']:.4f}")

    if CONFIG.get("use_wandb"):
        # Create summary table
        summary_table = wandb.Table(
            columns=["Fold", "Best Epoch", "Accuracy", "Precision", "Recall", "ROC-AUC"],
            data=[
                [r['fold'], r['best_epoch'], r['val_accuracy'], r['precision'], r['recall'], r['roc_auc']]
                for r in all_results
            ]
        )

        # Create a summary run to log the table
        with wandb.init(
            project=CONFIG["PROJECT_NAME"],
            name=f"{experiment_group_name}_summary",
            job_type="summary",
            group=experiment_group_name
        ) as summary_run:
            summary_run.log({"cross_validation_results": summary_table})
            summary_run.summary["mean_accuracy"] = np.mean(accuracies)
            summary_run.summary["std_accuracy"] = np.std(accuracies)
    else:
        print("W&B logging disabled; skipping summary upload.")
        print(f"Mean accuracy: {np.mean(accuracies):.4f}, Std: {np.std(accuracies):.4f}")


# ============================================================================
# RUN
# ============================================================================

if __name__ == '__main__':
    run_k_fold()